# Core PyTorch Mechanics: Autograd, Modules, and Training


## 1. Autograd and the Computational Graph

__What is a leaf tensor?__

A leaf tensor is a tensor at the beginning of the computational graph. It was not created by an operation tracked by autograd. By default, any tensor we create (e.g. `torch.tensor([1., 2.], requires_grad=True)` is a leaf. Parameters of neural networks (`nn.Parameter`) are also leaf tensors. Non-leaf tensors are the results of operations (`y = x + 2`).

__When does `.grad` populate?__

The `.grad` attribute for leaf tensors when you call `.backward()` on a scalar output (like loss) that depends on those leaves. By default, PyTorch frees the graph after `.backward()` and non-leaf tensors do not have their gradients saved to memory.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Creating Tensors

In [2]:
import numpy as np

# from python/numpy data
a = torch.tensor([1., 2., 3.])
b = torch.tensor(np.array([2., 3., 4.]))
c = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

# factory functions
zeros = torch.zeros(3, 4)   # shape is NOT a tuple
ones = torch.ones(3, 4, dtype=torch.float32)    # default for float is float32 due to GPU compatibility
rng = torch.rand(5, 5)      # standard normal
eye = torch.eye(3)          # identity


# from numpy
np_arr = np.arange(1, 5, dtype=float)
t_arr = torch.from_numpy(np_arr)        # shares memory w/ np_arr
t_arr2 = torch.tensor(np_arr)           # DOES NOT share memory w/ np_arr

t_arr[0] = 1000.

print(np_arr)

[1000.    2.    3.    4.]


### Data Types (`dtypes`)

```python
torch.float32   # default in torch
torch.float64
torch.float16
torch.bfloat16
torch.int32
torch.int64     # default in torch
torch.bool
```

In [3]:
x = torch.tensor([1, 2, 3])
print(x.dtype)

y = x.float()   # casts to float32
z = x.to(torch.float16) # explicit cast

torch.int64


### Shape, Stride, and Storage

A tensor's __storage__ is a contiguous, one-dimensional array of numbers in memory. Multiple tensors can share the same storage.

The __stride__ of a tensor is a tuple of integers. The _k_-th stride tells you how many elements you must skip to move one position along dimension _k_.

Given a $m \times n$ tensor, the element at position (i, j) is at storage offset:
    $offset = i * stride[0] + j*stride[1]$

To get to the next row's element (at same "place") need to move `stride[0]` places. Note that in a `mxn` matrix, `stride[0] = tensor.shape[-1]` - the number of columns.

In [12]:
x = torch.arange(1, 7, dtype=torch.float).reshape(2, 3)

print(x.shape)
print(x.stride())   # (3, 1)
# Moving along dim 0 (row): skip 3 elements
# Moving along dim 1 (col): skip 1 element


torch.Size([2, 3])
(3, 1)


## 2. Views vs. Copies

Many PyTorch operations return __views__: tensors that share the same underlying storage but with different shape/stride metadata.

In [15]:
a = torch.arange(12)
print(f"{a=}")

b = a.view(3, 4)            # view: shared storage
print(f"{b=}")

c = a.reshape(3, 4)          # view if possible, copy o.w.

b[0, 0] = 99

print(f"{a[0]=}")
print(f"{b[0]=}")
print(f"{c[0]=}")

# Transpose returns a view (non-contiguous)
t = torch.randn(3, 4)
print(f"{t=}")

t_T = t.T
print(f"{t_T=}")

print("\n")
print(f"{t.stride()=}")
print(f"{t_T.stride()=}")
print(f"{t_T.is_contiguous()=}")

a=tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
b=tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
a[0]=tensor(99)
b[0]=tensor([99,  1,  2,  3])
c[0]=tensor([99,  1,  2,  3])
t=tensor([[ 0.6781, -0.1067, -1.3776,  0.9754],
        [-0.0489, -0.7441,  0.1770, -1.1156],
        [ 0.2698, -1.1076,  0.3067,  0.9394]])
t_T=tensor([[ 0.6781, -0.0489,  0.2698],
        [-0.1067, -0.7441, -1.1076],
        [-1.3776,  0.1770,  0.3067],
        [ 0.9754, -1.1156,  0.9394]])


t.stride()=(4, 1)
t_T.stride()=(1, 4)
t_T.is_contiguous()=False


## 3. Advanced (Fancy) Indexing in NumPy and PyTorch

General idea is that each element in `idx` is treated as a position to "gather" (hint: gather is a function in pytorch) from. The output shape equals `idx.shape` (`a[idx].shape == idx.shape`). The core idea is that the index array is a __gather map__.

In [4]:
# 1D case
import numpy as np

a = np.array([10, 20, 30, 40, 50])
idx = np.array([0, 3, 3, 1])

a[idx]


array([10, 40, 40, 20])

When working w/ a 2D array with two index arrays, they pair up __element-wise__ (not as a grid.)

In [5]:
a = np.array([[0, 1, 2],
              [3, 4, 5],
              [6, 7, 8]])

rows = np.array([0, 1, 2])
cols = np.array([2, 0, 1])

a[rows, cols]

array([2, 3, 7])

Step by step, this gathers:

`a[0, 2]` -> 2
`a[1, 0]` -> 3
`a[2, 1]` -> 7


The two index arrays are zipped together, both must be the same shape or __broadcastable__.

__Broadcasting b/w Index Arrays__

If the index arrays have _different_ shapes, NumPy broadcasts them against each other first (if possible) and the output takes the broadcast shape.

In [6]:
a = np.array([[0, 1, 2],
              [3, 4, 5],
              [6, 7, 8]])

rows = np.array([0, 2])[:, np.newaxis]  # shape (2, 1)
cols = np.array([0, 1, 2])              # shape (3,)

result = a[rows, cols]
result

array([[0, 1, 2],
       [6, 7, 8]])

`rows` has shape `(2, 1)` and `cols` has shape `(3,)`
* `cols` gets broadcasted to `(1, 3)`
* `cols` gets broadcasted to `(2, 3)`
* `rows` gets broadcasted to `(2, 3)`

`rows` and `cols` now looks like:
```python
rows = np.array([[0, 0, 0],
                 [2, 2, 2]])

cols = np.array([[0, 1, 2],
                 [0, 1, 2]])
```

`result` will thus have shape `(2, 3)` where `result[i, j] = a[rows[i, j], cols[i, j]]

__Boolean Indexing__

A boolean array acts as a maks. Internally, NumPy converts it to `np.nonzero(mask)` -- a tuple of integer index arrays -- and then applies integer fancy indexing.

In [7]:
a = np.array([[1, 2], [3, 4], [5, 6]])

mask = a > 3

# what numpy does internally
rows, cols = np.nonzero(mask)
print(f"{a[rows, cols]=}")
print(f"{a[mask]=}")

a[rows, cols]=array([4, 5, 6])
a[mask]=array([4, 5, 6])


### Mixing Fancy and Basic Indexing

In [14]:
a = np.zeros((2, 3, 4, 5))

idx = np.array([0, 1])


print(f"{a[idx] == a}")


a[:, idx, idx, :].shape

[[[[ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]]

  [[ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]]

  [[ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]]]


 [[[ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]]

  [[ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]]

  [[ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]
   [ True  True  True  True  True]]]]


(2, 2, 5)

`a[idx]` grabs the first two elements of the 0-th dimension. The 0-th dimension has two things in it and we grab both so thus `a[idx] == a`


The rule is always: the indexed dimension gets replaced by the shape of the index array. In our case, they happened to coincide.

In [17]:
idx = np.array([0, 1])[np.newaxis, :]   # shape (1, 2)
print(f"{idx.shape=}")

a[idx].shape

idx.shape=(1, 2)


(1, 2, 3, 4, 5)

Dimension 0 of `a` (size 2) gets replaced by the shape of `idx` (`(1, 2)`) and then the remaining dimension `(3, 4, 5)` gets appended:
```python
(2, 3, 4, 5) -> replace 2 with (1, 2) -> (1, 2, 3, 4, 5)
```

### Scatter via Fancy Indexing (Assignment)

Fancy indexing works on the left side of `=` too. This is the "scatter" dual of "gather".

In [18]:
a = np.zeros(5)
idx = np.array([1, 2, 3])

a[idx] += 1
a

array([0., 1., 1., 1., 0.])

In [19]:
b = np.zeros(5)
idx = np.array([1, 3, 3])
b[idx] += 1
b

array([0., 1., 0., 1., 0.])

Note that in `b`, when we do `b[idx] += 1`, index 3 does not accumulate. The `+=` is not atomic. NumPy just buffers the results and writes once. If we need accumulation, we use `np.add.at(a, idx, 1)`

In [20]:
c = np.zeros(5)
idx = np.array([1, 3, 3])
np.add.at(c, idx, 1)

c

array([0., 1., 0., 2., 0.])

### Now in PyTorch

__1D indexing__ is identical to NumPy

In [21]:
import torch

a = torch.tensor([10, 20, 30, 40, 50])
idx = torch.tensor([0, 3, 3, 1])

a[idx]

tensor([10, 40, 40, 20])

__Multi-dimensional Gather__ (NumPy style) works in PyTorch too. Same zip semantics as NumPy: `result[i] = a[rows[i], cols[i]]`

In [22]:
a = torch.tensor([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])

rows = torch.tensor([0, 1, 2])
cols = torch.tensor([2, 0, 1])

a[rows, cols]

tensor([3, 4, 8])

__Broadcasting__ between Index Tensors

In [23]:
a = torch.tensor([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])

rows = torch.tensor([0, 2]).unsqueeze(1)    # shape (2, 1)
cols = torch.tensor([2, 0, 1])              # shape (3, )

a[rows, cols]

tensor([[3, 1, 2],
        [9, 7, 8]])

`.unsqueeze(1)` is the equivalent of `[:, np.newaxis]` in NumPy.

__Boolean Masking__: Same as NumPy

Always flattened because the number of `True` entries isn't known at compile time.

In [24]:
a = torch.tensor([[1, 2], [3, 4], [5, 6]])

mask = a > 3

a[mask]

tensor([4, 5, 6])

__`torch.gather`__

`gather` pins all non-dim coordinates to the inex tensor's own position.

In [29]:
a = torch.tensor([[10, 20, 30],
                  [40, 50, 60]])

index1 = torch.tensor([[2, 0],
                      [1, 2]])

# "For each row, pick these columns"
out1 = torch.gather(a, dim=1, index=index1)
print(f"{out1=}")

# For each col, pick these rows
index2 = torch.tensor([[1, 0, 0],
                       [0, 1, 1],
                       [1, 1, 1]])

out2 = torch.gather(a, dim=0, index=index2)
print(f"{out2=}")

out1=tensor([[30, 10],
        [50, 60]])
out2=tensor([[40, 20, 30],
        [10, 50, 60],
        [40, 50, 60]])


```python
# for dim=1
output[i, j] = a[i, index[i, j]]

# for dim=0
output[i, j] = a[index[i, j], j]
```

__`scatter`__ The write-side dual of `gather`

`scatter_` is the inverse - instead of "read from these positions", it's "write to these positions"

In [30]:
dest = torch.zeros(2, 3, dtype=torch.float)
index = torch.tensor([[0, 2],
                      [1, 0]])

src = torch.tensor([[99., 88.],
                    [77., 66.]])

dest.scatter_(dim=1, index=index, src=src)

tensor([[99.,  0., 88.],
        [66., 77.,  0.]])

What the indexing looks like.
```python
# for dim=1
dest[i, index[i, j]] = src[i, j]


# for dim=0
dest[index[i, j], j] = src[i, j]
```

Just like in Numpy, accumulation does not occur, need to specify with `scatter_add_`

In [31]:
a = torch.zeros(5)
idx = torch.tensor([1, 3, 3])
vals = torch.ones(3)

a.scatter_add_(0, idx, vals)
a

tensor([0., 1., 0., 2., 0.])

__`index_select`__ Simpler row/col selection

In [32]:
a = torch.tensor([[10, 20, 30],
                  [40, 50, 60],
                  [70, 80, 90]])

torch.index_select(a, dim=0, index=torch.tensor([0, 2]))                

tensor([[10, 20, 30],
        [70, 80, 90]])